In [ ]:
# Step 1: Load and inspect the cleaned dataset
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "..")
import pandas as pd
import numpy as np

# Load the cleaned dataset
cleaned_df = pd.read_csv(
    "../data/processed/cleaned_farmgate_prices.csv",
    parse_dates=["Date"]
)

# Create a separate dataset for forecasting
model_data = cleaned_df.copy()

# Verify the dataset
print("Total records:", len(model_data))
print("Total commodities:", model_data["Commodity"].nunique())
print("Total categories:", model_data["Category"].nunique())
print("Missing prices:", model_data["Price"].isna().sum())
print("Interpolated prices:", model_data["Was_Interpolated"].sum())


In [ ]:
# Step 2: Identify all commodities available for forecasting

# Get all unique category and commodity combinations
commodities = model_data[
    ["Category", "Commodity"]
].drop_duplicates().reset_index(drop=True)

# Count commodities in each agricultural category
category_counts = (
    commodities.groupby("Category")
    .size()
    .reset_index(name="Commodity Count")
)

print("Total Commodities:", len(commodities))

# Display the number of commodities per category
display(category_counts)


# Display all 58 commodities organized by category
print("\nComplete List of Agricultural Commodities:")

display(
    commodities.sort_values(
        by=["Category", "Commodity"]
    ).reset_index(drop=True)
)

In [3]:
# Step 3: Set up the reusable model evaluation pipeline
from src.evaluation import evaluate_commodity

# Set to True only when we need to rerun all model evaluations
run_full_evaluation = False

print("Evaluation function imported successfully.")
print("Run full evaluation:", run_full_evaluation)

Evaluation function imported successfully.
Run full evaluation: False


In [ ]:
# Step 4: Evaluate all commodities or reuse saved evaluation results
from pathlib import Path

if run_full_evaluation:
    # Store evaluation results and errors
    all_results = []
    failed_commodities = []

    # Evaluate all 58 commodities using our reusable function
    for category, commodity in commodities.itertuples(index=False, name=None):
        try:
            results = evaluate_commodity(model_data, category, commodity)

            # Identify the commodity for each evaluation
            results["Category"] = category
            results["Commodity"] = commodity

            all_results.append(results)
            print(f"Completed: {commodity}")

        except Exception as error:
            failed_commodities.append((category, commodity, str(error)))
            print(f"Failed: {commodity} - {error}")

    # Combine evaluation results into one table
    all_results_df = pd.concat(all_results, ignore_index=True)

    # Save new results only if all 58 commodities were evaluated
    if len(all_results) == 58 and not failed_commodities:
        all_results_df.to_csv(
            "../data/processed/model_evaluation_results_rerun.csv",
            index=False
        )
        print("New evaluation results saved successfully.")
    else:
        print("Incomplete evaluations. Results were not saved.")


else:
    # Reuse previously calculated results without retraining
    evaluation_file = "../data/processed/model_evaluation_results.csv"

    if not Path(evaluation_file).exists():
        raise FileNotFoundError(
            "Evaluation CSV not found. Restore the file or enable full evaluation."
        )

    all_results_df = pd.read_csv(evaluation_file)
    print("Loaded existing evaluation results.")

# Display the results
print("Total Model Evaluations:", len(all_results_df))
print("Commodities Evaluated:", all_results_df["Commodity"].nunique())

display(all_results_df.head(20).round(2))


In [ ]:
# Step 5: Select the best forecasting model for each commodity

# Rank models by lowest MAE, then RMSE
ranked_results = all_results_df.sort_values(["MAE", "RMSE"])

# Select the best-performing model for each commodity
best_models_df = ranked_results.drop_duplicates(
    subset=["Category", "Commodity"],
    keep="first"
)

# Keep only the important columns
best_models_df = best_models_df[
    ["Category", "Commodity", "Model", "MAE", "RMSE"]
].reset_index(drop=True)

# Display the selected models for all 58 commodities
print("Commodities with selected models:", len(best_models_df))
display(best_models_df.round(2))


In [ ]:
# Step 6: Test the reusable forecasting function
from src.forecasting import forecast_commodity

# Select one Rootcrops commodity to test the function
test_commodity = best_models_df[
    best_models_df["Category"] == "Rootcrops"
].iloc[0]

# Train its selected model and generate six-month forecasts
test_forecasts, test_fitted_model = forecast_commodity(
    model_data,
    test_commodity["Category"],
    test_commodity["Commodity"],
    best_models_df
)

# Verify the results
print("Commodity:", test_commodity["Commodity"])
print("Selected Model:", test_commodity["Model"])
print("Model Is Fitted:", test_fitted_model.is_fitted)
print("Forecast Months:", len(test_forecasts))

display(test_forecasts.round(2))

In [7]:

# Step 8: Generate six-month forecasts for all 58 commodities

from src.forecasting import forecast_commodity

# Store forecasts and any errors
all_forecasts = []
failed_commodities = []

# Forecast each commodity using its selected best model
for category, commodity in commodities.itertuples(index=False, name=None):
    try:
        forecasts, fitted_model = forecast_commodity(
            model_data, category, commodity, best_models_df
        )

        all_forecasts.append(forecasts)
        print(f"Completed: {commodity}")

    except Exception as error:
        failed_commodities.append((commodity, str(error)))
        print(f"Failed: {commodity} - {error}")

# Combine the six-month forecasts into one table
all_forecasts_df = pd.concat(all_forecasts, ignore_index=True)

print("\nCommodities Forecasted:", len(all_forecasts))
print("Total Forecasts:", len(all_forecasts_df))
print("Failed Commodities:", len(failed_commodities))

display(all_forecasts_df.head(12).round(2))


Completed: Habitchuelas [Snap beans]
Completed: Mongo [Mungbean], green, labo
Completed: Mongo [Mungbean], green, shiny
Completed: Mongo [Mungbean], yellow, shiny
Completed: Peanut, with shell, dry
Completed: Peanut, with shell, fresh
Completed: Sitao [Stringbeans]
Completed: Sweet peas, Baguio (Chinese)
Completed: Corngrain [Maize] White, matured
Completed: Corngrain [Maize] Yellow, matured
Completed: Green Corn (Maize, green), White
Completed: Palay [Paddy] Other Variety, dry (conv. to 14% mc)
Completed: Chili pepper fruit (Siling labuyo)
Completed: Garlic, dried bulb
Completed: Ginger, Hawaiian
Completed: Ginger, Native
Completed: Onion Leeks
Completed: Onion, red shallot (Sibuyas Tagalog)
Completed: Paminta [Black pepper]
Completed: Pepper, bell
Completed: Pepper, finger
Completed: Spring onion
Completed: Ampalaya fruit [Bitter gourd]
Completed: Cucumber
Completed: Eggplant, long, purple
Completed: Eggplant, native, round
Completed: Okra [Lady's finger]
Completed: Papaya, green
Com

,Category,Commodity,Last Observed Date,Last Observed Price,Forecast Date,Forecast Price,Model,MAE,RMSE
0,Beans and Legumes,Habitchuelas [Snap beans],2026-06,41.82,2026-07,35.53,Holt-Winters,10.37,14.31
1,Beans and Legumes,Habitchuelas [Snap beans],2026-06,41.82,2026-08,40.19,Holt-Winters,10.37,14.31
2,Beans and Legumes,Habitchuelas [Snap beans],2026-06,41.82,2026-09,41.08,Holt-Winters,10.37,14.31
3,Beans and Legumes,Habitchuelas [Snap beans],2026-06,41.82,2026-10,46.21,Holt-Winters,10.37,14.31
4,Beans and Legumes,Habitchuelas [Snap beans],2026-06,41.82,2026-11,47.53,Holt-Winters,10.37,14.31
5,Beans and Legumes,Habitchuelas [Snap beans],2026-06,41.82,2026-12,47.82,Holt-Winters,10.37,14.31
6,Beans and Legumes,"Mongo [Mungbean], green, labo",2026-06,77.20,2026-07,77.26,AutoARIMA,6.02,10.86
7,Beans and Legumes,"Mongo [Mungbean], green, labo",2026-06,77.20,2026-08,77.41,AutoARIMA,6.02,10.86
8,Beans and Legumes,"Mongo [Mungbean], green, labo",2026-06,77.20,2026-09,77.56,AutoARIMA,6.02,10.86
9,Beans and Legumes,"Mongo [Mungbean], green, labo",2026-06,77.20,2026-10,77.71,AutoARIMA,6.02,10.86


In [8]:

# Step 8: Load existing six-month forecasts or generate them

from pathlib import Path
from src.forecasting import forecast_commodity

forecast_file = Path("../data/processed/future_six_month_forecasts.csv")

if forecast_file.exists():
    # Reuse existing forecast results without retraining
    all_forecasts_df = pd.read_csv(forecast_file)
    print("Loaded existing forecast results.")

else:
    # Generate six-month forecasts for all 58 commodities
    all_forecasts = []
    failed_commodities = []

    for category, commodity in commodities.itertuples(index=False, name=None):
        try:
            forecasts, fitted_model = forecast_commodity(
                model_data,
                category,
                commodity,
                best_models_df
            )

            all_forecasts.append(forecasts)
            print(f"Completed: {commodity}")

        except Exception as error:
            failed_commodities.append((commodity, str(error)))
            print(f"Failed: {commodity} - {error}")

    # Make sure at least one forecast was generated
    if not all_forecasts:
        raise RuntimeError("No forecasts were generated.")

    # Combine forecasts after processing all commodities
    all_forecasts_df = pd.concat(all_forecasts, ignore_index=True)

    # Verify every commodity has exactly six forecast months
    forecast_counts = all_forecasts_df.groupby(
        ["Category", "Commodity"]
    ).size()

    # Save only when all 348 forecasts are complete
    if len(forecast_counts) == 58 and forecast_counts.eq(6).all() and not failed_commodities:
        all_forecasts_df.to_csv(forecast_file, index=False)
        print("All 348 forecasts saved successfully.")
    else:
        print("Incomplete forecasts. Results were not saved.")

# Show summary
print("Commodities Forecasted:", all_forecasts_df["Commodity"].nunique())
print("Total Forecast Rows:", len(all_forecasts_df))

display(all_forecasts_df.head(12).round(2))


Loaded existing forecast results.
Commodities Forecasted: 58
Total Forecast Rows: 348


,Category,Commodity,Last Observed Date,Last Observed Price,Forecast Date,Forecast Price,Model,MAE,RMSE
0,Beans and Legumes,Habitchuelas [Snap beans],2026-06,41.82,2026-07,35.53,Holt-Winters,10.37,14.31
1,Beans and Legumes,Habitchuelas [Snap beans],2026-06,41.82,2026-08,40.19,Holt-Winters,10.37,14.31
2,Beans and Legumes,Habitchuelas [Snap beans],2026-06,41.82,2026-09,41.08,Holt-Winters,10.37,14.31
3,Beans and Legumes,Habitchuelas [Snap beans],2026-06,41.82,2026-10,46.21,Holt-Winters,10.37,14.31
4,Beans and Legumes,Habitchuelas [Snap beans],2026-06,41.82,2026-11,47.53,Holt-Winters,10.37,14.31
5,Beans and Legumes,Habitchuelas [Snap beans],2026-06,41.82,2026-12,47.82,Holt-Winters,10.37,14.31
6,Beans and Legumes,"Mongo [Mungbean], green, labo",2026-06,77.20,2026-07,77.26,AutoARIMA,6.02,10.86
7,Beans and Legumes,"Mongo [Mungbean], green, labo",2026-06,77.20,2026-08,77.41,AutoARIMA,6.02,10.86
8,Beans and Legumes,"Mongo [Mungbean], green, labo",2026-06,77.20,2026-09,77.56,AutoARIMA,6.02,10.86
9,Beans and Legumes,"Mongo [Mungbean], green, labo",2026-06,77.20,2026-10,77.71,AutoARIMA,6.02,10.86


In [9]:
# Step 9: Validate the six-month forecasts for all 58 commodities

# Count the forecast months for each commodity
forecast_counts = all_forecasts_df.groupby(
    ["Category", "Commodity"]
).size()

# Check for missing values
missing_values = all_forecasts_df.isna().sum().sum()

# Check for duplicate commodity-month forecasts
duplicate_forecasts = all_forecasts_df.duplicated(
    subset=["Category", "Commodity", "Forecast Date"]
).sum()

# Display validation results
print("Total Forecasts:", len(all_forecasts_df))
print("Commodities:", len(forecast_counts))
print("All commodities have 6 forecasts:", forecast_counts.eq(6).all())
print("Missing Values:", missing_values)
print("Duplicate Forecasts:", duplicate_forecasts)


Total Forecasts: 348
Commodities: 58
All commodities have 6 forecasts: True
Missing Values: 0
Duplicate Forecasts: 0
